# ML_20252 - Wine Price EDA

This notebook implements the EDA phase only. Training sections are intentionally left out so model design can be decided after reviewing the data.

## 0. Setup

In [1]:
from pathlib import Path
import warnings

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
REFERENCE_YEAR = 2026

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
OUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"
DOCS_DIR = PROJECT_DIR / "docs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Outputs:", OUT_DIR)

Project: D:\tai ve\VISUAL CODE STUDIO\ML-20252\ML_20252
Data: D:\tai ve\VISUAL CODE STUDIO\ML-20252\ML_20252\data
Outputs: D:\tai ve\VISUAL CODE STUDIO\ML-20252\ML_20252\outputs


## 1. Load Data

In [2]:
raw = pd.read_csv(DATA_DIR / "raw_wines.csv", encoding="utf-8-sig")
cleaned = pd.read_csv(DATA_DIR / "cleaned_wines.csv", encoding="utf-8-sig")
processed = pd.read_csv(DATA_DIR / "processed_wines.csv", encoding="utf-8-sig")

print("raw:", raw.shape)
print("cleaned:", cleaned.shape)
print("processed:", processed.shape)
display(raw.head())

raw: (200, 16)
cleaned: (199, 22)
processed: (199, 10)


,url,product_name,sku,price_vnd,grape_variety,wine_type,brand,origin_country,alcohol_content,volume,region,vintage,rating_score,rating_count,image_url,short_description
0,https://winecellar.vn/sample-wine-0001,Domaine Sample Chardonnay 2012,SAMPLE-0001,723000,Chardonnay,Rượu Vang Trắng,Domaine Sample,Pháp,12.9,750.0,Champagne,2012.0,4.5,142,NaN,Dữ liệu mẫu cho rượu vang trắng từ Pháp.
1,https://winecellar.vn/sample-wine-0002,Estate Select Cabernet Sauvignon 2014,SAMPLE-0002,762000,Cabernet Sauvignon,Rượu Vang Ngọt,Estate Select,Ý,13.0,750.0,Veneto,2014.0,4.5,142,NaN,Dữ liệu mẫu cho rượu vang ngọt từ Ý.
2,https://winecellar.vn/sample-wine-0003,Casa Vista Malbec 2016,SAMPLE-0003,507000,Malbec,Rượu Vang Đỏ,Casa Vista,Argentina,12.7,750.0,Mendoza,2016.0,3.6,89,NaN,Dữ liệu mẫu cho rượu vang đỏ từ Argentina.
3,https://winecellar.vn/sample-wine-0004,Grand Valley Cabernet Sauvignon 2022,SAMPLE-0004,673000,Cabernet Sauvignon,Champagne,Grand Valley,Úc,12.3,375.0,Margaret River,2022.0,4.2,99,NaN,Dữ liệu mẫu cho champagne từ Úc.
4,https://winecellar.vn/sample-wine-0005,Chateau Demo Merlot 2015,SAMPLE-0005,1224000,Merlot,Champagne,Chateau Demo,Úc,14.9,750.0,Barossa Valley,2015.0,4.6,23,NaN,Dữ liệu mẫu cho champagne từ Úc.


## 2. Data Audit

In [3]:
def audit_frame(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(dropna=True),
    }).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

print("Raw data audit")
display(audit_frame(raw))

print("Cleaned data audit")
display(audit_frame(cleaned))

Raw data audit


,dtype,missing,missing_pct,n_unique
image_url,float64,200,100.0,0
vintage,float64,15,7.5,15
url,object,0,0.0,200
sku,object,0,0.0,200
price_vnd,int64,0,0.0,185
product_name,object,0,0.0,180
rating_count,int64,0,0.0,107
short_description,object,0,0.0,54
alcohol_content,float64,0,0.0,51
region,object,0,0.0,17


Cleaned data audit


,dtype,missing,missing_pct,n_unique
image_url,float64,199,100.0,0
url,object,0,0.0,199
sku,object,0,0.0,199
price_vnd,int64,0,0.0,184
product_name,object,0,0.0,179
rating_count,int64,0,0.0,107
short_description,object,0,0.0,54
alcohol_content,float64,0,0.0,51
region,object,0,0.0,17
region_enc,int64,0,0.0,17


In [4]:
price = pd.to_numeric(cleaned["price_vnd"], errors="coerce")
price_summary = price.describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).to_frame("price_vnd")
price_summary.loc["log1p_skew", "price_vnd"] = np.log1p(price.dropna()).skew()

print("Target summary")
display(price_summary.round(3))

categorical_cols = ["wine_type", "grape_variety", "origin_country", "region", "brand", "volume"]
for col in categorical_cols:
    if col in cleaned.columns:
        print(f"\nTop values for {col}")
        display(cleaned[col].value_counts(dropna=False).head(10).to_frame("count"))

Target summary


,price_vnd
count,199.000
mean,778773.869
std,347432.115
min,200000.000
1%,241960.000
5%,373900.000
25%,538500.000
50%,696000.000
75%,930000.000
95%,1593000.000



Top values for wine_type


,count
wine_type,
Rượu Vang Organic,32
Rượu Vang Hồng,30
Rượu Vang Sủi,25
Rượu Vang Trắng,24
Rượu Vang Đỏ,24
Rượu Vang Không Cồn,24
Champagne,20
Rượu Vang Ngọt,20



Top values for grape_variety


...

,count
region,
Margaret River,19
Salta,18
Maipo Valley,17
Barossa Valley,17
Tuscany,16
Sonoma,15
Ribera Del Duero,14
Mendoza,13
Colchagua Valley,11



Top values for brand


,count
brand,
Reserve Cellar,38
Estate Select,37
Casa Vista,36
Chateau Demo,31
Domaine Sample,29
Grand Valley,28



Top values for volume


,count
volume,
750.0,158
375.0,25
1500.0,16


## 3. EDA

### 3.1 Target Distribution

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(cleaned["price_vnd"] / 1e6, bins=25, kde=True, ax=axes[0], color="#4C78A8")
axes[0].set_title("Price Distribution")
axes[0].set_xlabel("Price (million VND)")
axes[0].set_ylabel("Count")

sns.histplot(np.log1p(cleaned["price_vnd"]), bins=25, kde=True, ax=axes[1], color="#F58518")
axes[1].set_title("log1p(price_vnd) Distribution")
axes[1].set_xlabel("log1p(price_vnd)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(OUT_DIR / "notebook_eda_price_distribution.png", dpi=140, bbox_inches="tight")
plt.show()

### 3.2 Price By Wine Attributes

In [6]:
cat_cols_for_eda = ["wine_type", "origin_country", "grape_variety", "brand", "volume"]

fig, axes = plt.subplots(len(cat_cols_for_eda), 1, figsize=(12, 4 * len(cat_cols_for_eda)))
for ax, col in zip(axes, cat_cols_for_eda):
    order = cleaned.groupby(col)["price_vnd"].median().sort_values(ascending=False).index
    sns.boxplot(data=cleaned, x=col, y="price_vnd", order=order, ax=ax)
    ax.set_title(f"Price by {col}")
    ax.set_xlabel("")
    ax.set_ylabel("Price (VND)")
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig(OUT_DIR / "notebook_eda_category_price.png", dpi=140, bbox_inches="tight")
plt.show()

In [7]:
for col in cat_cols_for_eda:
    summary = (
        cleaned.groupby(col, dropna=False)["price_vnd"]
        .agg(count="count", median="median", mean="mean")
        .sort_values("median", ascending=False)
        .round(0)
    )
    print(f"\nPrice summary by {col}")
    display(summary)


Price summary by wine_type


,count,median,mean
wine_type,,,
Champagne,20,1099500.0,1135300.0
Rượu Vang Sủi,25,821000.0,819680.0
Rượu Vang Organic,32,815000.0,882719.0
Rượu Vang Trắng,24,705000.0,822000.0
Rượu Vang Đỏ,24,686500.0,741333.0
Rượu Vang Hồng,30,615500.0,651933.0
Rượu Vang Ngọt,20,592000.0,668850.0
Rượu Vang Không Cồn,24,493500.0,544833.0



Price summary by origin_country


,count,median,mean
origin_country,,,
Pháp,19,953000.0,1021474.0
Ý,34,845500.0,898088.0
Mỹ,26,805000.0,922000.0
Úc,36,700000.0,775667.0
Tây Ban Nha,25,636000.0,692160.0
Chile,28,599500.0,617571.0
Argentina,31,523000.0,598097.0



Price summary by grape_variety


,count,median,mean
grape_variety,,,
Chardonnay,23,727000.0,846870.0
Merlot,32,726500.0,832469.0
Malbec,21,721000.0,749762.0
Syrah,30,696500.0,747000.0
Riesling,25,684000.0,773760.0
Cabernet Sauvignon,20,673500.0,746400.0
Sauvignon Blanc,24,662000.0,689792.0
Pinot Noir,24,647000.0,828208.0



Price summary by brand


,count,median,mean
brand,,,
Chateau Demo,31,821000.0,831968.0
Casa Vista,36,707500.0,805889.0
Grand Valley,28,698000.0,730286.0
Reserve Cellar,38,693000.0,748553.0
Domaine Sample,29,687000.0,769103.0
Estate Select,37,677000.0,783135.0



Price summary by volume


,count,median,mean
volume,,,
1500.0,16,1288000.0,1378438.0
750.0,158,695500.0,757582.0
375.0,25,465000.0,528920.0


### 3.3 Numeric Relationships

In [8]:
numeric_cols = ["price_vnd", "vintage", "alcohol_content", "volume", "rating_score", "rating_count"]
numeric_cols = [c for c in numeric_cols if c in cleaned.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cleaned[numeric_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[0])
axes[0].set_title("Numeric Correlation")

sns.scatterplot(data=cleaned, x="rating_score", y="price_vnd", hue="wine_type", ax=axes[1])
axes[1].set_title("Rating Score vs Price")
axes[1].set_xlabel("Rating score")
axes[1].set_ylabel("Price (VND)")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Wine type")

plt.tight_layout()
plt.savefig(OUT_DIR / "notebook_eda_numeric_relationships.png", dpi=140, bbox_inches="tight")
plt.show()

In [9]:
quality_checks = pd.DataFrame({
    "check": [
        "raw rows",
        "cleaned rows",
        "processed rows",
        "raw missing vintage pct",
        "cleaned missing image_url pct",
        "cleaned alcohol <= 0 rows",
        "cleaned alcohol > 0.20 rows",
    ],
    "value": [
        len(raw),
        len(cleaned),
        len(processed),
        round(raw["vintage"].isna().mean() * 100, 1),
        round(cleaned["image_url"].isna().mean() * 100, 1),
        int((cleaned["alcohol_content"] <= 0).sum()),
        int((cleaned["alcohol_content"] > 0.20).sum()),
    ],
})

display(quality_checks)

,check,value
0,raw rows,200.0
1,cleaned rows,199.0
2,processed rows,199.0
3,raw missing vintage pct,7.5
4,cleaned missing image_url pct,100.0
5,cleaned alcohol <= 0 rows,1.0
6,cleaned alcohol > 0.20 rows,11.0


### 3.4 Data Quality Notes

- Use `price_vnd` as a regression target.
- Drop `image_url` because it is fully missing.
- Drop identifiers (`url`, `sku`) from training.
- Keep `rating_count`; current processed data omits it, but it may explain confidence/popularity.
- Validate alcohol values near `0` or above `0.20` after normalization because those are unlikely for normal wine.
- Prefer one-hot encoding for nominal categories instead of label encoding.

Stop point: feature engineering and model training are intentionally not implemented yet.

## 4. Modeling Plan

This section compares two feature sets with the same split, same candidate models, and same metrics:

1. `BaselineOriginal`: all usable original dataset features, with only cleaning and encoding.
2. `EngineeredFeatures`: a slim v2 engineered set designed after the first evaluation. It keeps stable low-cardinality and numeric transformations, and removes sparse interaction/target-stat features that overfit on a 200-row dataset.

The goal is to measure whether the slimmer engineered features improve MAE/RMSE/R2 over the original-feature baseline.

## 5. BaselineOriginal With Usable Original Features

The baseline uses all original predictors that are usable for modeling. It excludes only `url`, `sku`, and `image_url`: identifiers are near-unique per row, and `image_url` is fully missing.

In [10]:
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def normalize_text_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        out[col] = out[col].fillna("Unknown").astype(str).str.strip()
    return out


def prepare_original_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["price_vnd"] = pd.to_numeric(out["price_vnd"], errors="coerce")
    out = out[out["price_vnd"].notna() & (out["price_vnd"] > 0)].copy()

    for col in ["vintage", "alcohol_content", "volume", "rating_score", "rating_count"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    pct_mask = out["alcohol_content"] > 1
    out.loc[pct_mask, "alcohol_content"] = out.loc[pct_mask, "alcohol_content"] / 100
    out.loc[(out["alcohol_content"] <= 0) | (out["alcohol_content"] > 0.25), "alcohol_content"] = np.nan

    text_cols = ["product_name", "short_description"]
    cat_cols = ["grape_variety", "wine_type", "brand", "origin_country", "region"]
    out = normalize_text_columns(out, text_cols + cat_cols)
    out[cat_cols] = out[cat_cols].apply(lambda s: s.str.title())
    out["combined_text"] = out["product_name"] + " " + out["short_description"]
    return out


def build_preprocess(numeric_features: list[str], categorical_features: list[str], text_feature: str | None = None):
    transformers = [
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2)),
        ]), categorical_features),
    ]
    if text_feature is not None:
        transformers.append((
            "text",
            TfidfVectorizer(max_features=40, ngram_range=(1, 2), lowercase=True),
            text_feature,
        ))
    return ColumnTransformer(transformers, remainder="drop", sparse_threshold=0.0)


model_df = prepare_original_frame(raw)
target_col = "price_vnd"
y = model_df[target_col]
price_bins = pd.qcut(y, q=5, duplicates="drop", labels=False)

train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=price_bins,
)

baseline_numeric_features = [
    "alcohol_content",
    "volume",
    "vintage",
    "rating_score",
    "rating_count",
]
baseline_categorical_features = [
    "grape_variety",
    "wine_type",
    "brand",
    "origin_country",
    "region",
]
baseline_text_feature = "combined_text"
excluded_original_columns = ["url", "sku", "image_url"]

baseline_preprocess = build_preprocess(
    baseline_numeric_features,
    baseline_categorical_features,
    baseline_text_feature,
)

baseline_feature_cols = baseline_numeric_features + baseline_categorical_features + [baseline_text_feature]
X_base = model_df[baseline_feature_cols]
y_base = model_df[target_col]
X_base_train = X_base.loc[train_idx]
X_base_test = X_base.loc[test_idx]
y_train = y_base.loc[train_idx]
y_test = y_base.loc[test_idx]

print("Rows after target filtering:", len(model_df))
print("Train rows:", len(train_idx), "Test rows:", len(test_idx))
print("Baseline train:", X_base_train.shape, "Baseline test:", X_base_test.shape)
print("Baseline numeric:", baseline_numeric_features)
print("Baseline categorical:", baseline_categorical_features)
print("Baseline text:", baseline_text_feature)
print("Excluded:", excluded_original_columns)

Rows after target filtering: 200
Train rows: 160 Test rows: 40
Baseline train: (160, 11) Baseline test: (40, 11)
Baseline numeric: ['alcohol_content', 'volume', 'vintage', 'rating_score', 'rating_count']
Baseline categorical: ['grape_variety', 'wine_type', 'brand', 'origin_country', 'region']
Baseline text: combined_text
Excluded: ['url', 'sku', 'image_url']


## 6. EngineeredFeatures v2 Slim

This version removes high-cardinality interaction features and group target-stat encodings. It adds only stable numeric transforms, missingness flags, text flags, and low-cardinality bins.

In [11]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["is_missing_vintage"] = out["vintage"].isna().astype(int)
    out["is_missing_alcohol"] = out["alcohol_content"].isna().astype(int)

    out["alcohol_pct"] = out["alcohol_content"] * 100
    out["alcohol_diff_from_median"] = out["alcohol_content"] - out["alcohol_content"].median()
    out["alcohol_bin"] = pd.cut(
        out["alcohol_pct"],
        bins=[0, 10, 12, 13.5, 15, 100],
        labels=["very_low", "low", "medium", "high", "very_high"],
        include_lowest=True,
    )

    out["log_volume"] = np.log1p(out["volume"])

    out["wine_age"] = REFERENCE_YEAR - out["vintage"]
    out.loc[out["wine_age"] < 0, "wine_age"] = np.nan
    out["age_group"] = pd.cut(
        out["wine_age"],
        bins=[-1, 2, 5, 10, 20, 100],
        labels=["0-2", "3-5", "6-10", "11-20", "20+"],
    )

    out["log_rating_count"] = np.log1p(out["rating_count"])
    out["rating_bin"] = pd.cut(
        out["rating_score"],
        bins=[0, 3.5, 4.0, 4.3, 4.6, 5.0],
        labels=["low", "mid", "good", "very_good", "excellent"],
        include_lowest=True,
    )
    global_rating_mean = out["rating_score"].mean()
    rating_count_median = out["rating_count"].median()
    out["bayesian_rating"] = (
        out["rating_count"] / (out["rating_count"] + rating_count_median) * out["rating_score"]
        + rating_count_median / (out["rating_count"] + rating_count_median) * global_rating_mean
    )
    out["rating_x_popularity"] = out["rating_score"] * np.log1p(out["rating_count"])

    red_grapes = ["Merlot", "Syrah", "Malbec", "Cabernet Sauvignon", "Pinot Noir"]
    white_grapes = ["Chardonnay", "Riesling", "Sauvignon Blanc"]
    out["grape_color_group"] = np.select(
        [out["grape_variety"].isin(red_grapes), out["grape_variety"].isin(white_grapes)],
        ["red_grape", "white_grape"],
        default="other",
    )

    out["text_has_year"] = out["product_name"].str.contains(r"\b(?:19|20)\d{2}\b", regex=True).astype(int)
    out["description_length"] = out["short_description"].str.len()

    return out


engineered_df = add_engineered_features(model_df)

engineered_numeric_features = [
    "alcohol_content",
    "volume",
    "rating_score",
    "log_volume",
    "wine_age",
    "log_rating_count",
    "bayesian_rating",
    "rating_x_popularity",
    "alcohol_diff_from_median",
    "description_length",
]
engineered_binary_features = [
    "is_missing_vintage",
    "is_missing_alcohol",
    "text_has_year",
]
engineered_categorical_features = baseline_categorical_features + [
    "grape_color_group",
    "rating_bin",
    "age_group",
    "alcohol_bin",
]

removed_from_v1 = [
    "country_x_type",
    "type_x_volume",
    "brand_x_type",
    "country_x_grape",
    "type_x_alcohol_bin",
    "target-stat encodings",
    "manual price tiers",
    "duplicate binary category flags",
]

engineered_feature_cols = (
    engineered_numeric_features
    + engineered_binary_features
    + engineered_categorical_features
    + [baseline_text_feature]
)

X_eng = engineered_df[engineered_feature_cols]
X_eng_train = X_eng.loc[train_idx]
X_eng_test = X_eng.loc[test_idx]

engineered_preprocess = build_preprocess(
    engineered_numeric_features + engineered_binary_features,
    engineered_categorical_features,
    baseline_text_feature,
)

print("Engineered v2 train:", X_eng_train.shape, "Engineered v2 test:", X_eng_test.shape)
print("Engineered numeric:", engineered_numeric_features)
print("Engineered binary:", engineered_binary_features)
print("Engineered categorical:", engineered_categorical_features)
print("Removed from v1:", removed_from_v1)

Engineered v2 train: (160, 23) Engineered v2 test: (40, 23)
Engineered numeric: ['alcohol_content', 'volume', 'rating_score', 'log_volume', 'wine_age', 'log_rating_count', 'bayesian_rating', 'rating_x_popularity', 'alcohol_diff_from_median', 'description_length']
Engineered binary: ['is_missing_vintage', 'is_missing_alcohol', 'text_has_year']
Engineered categorical: ['grape_variety', 'wine_type', 'brand', 'origin_country', 'region', 'grape_color_group', 'rating_bin', 'age_group', 'alcohol_bin']
Removed from v1: ['country_x_type', 'type_x_volume', 'brand_x_type', 'country_x_grape', 'type_x_alcohol_bin', 'target-stat encodings', 'manual price tiers', 'duplicate binary category flags']


## 7. Shared Candidate Models And Evaluation

In [12]:
def evaluate_predictions(y_true, y_pred) -> dict:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    }


def build_models(preprocess):
    return {
        "DummyMedian": Pipeline([
            ("preprocess", preprocess),
            ("model", DummyRegressor(strategy="median")),
        ]),
        "RidgeLogTarget": TransformedTargetRegressor(
            regressor=Pipeline([
                ("preprocess", preprocess),
                ("model", Ridge(alpha=10.0)),
            ]),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        "ElasticNetLogTarget": TransformedTargetRegressor(
            regressor=Pipeline([
                ("preprocess", preprocess),
                ("model", ElasticNet(alpha=0.01, l1_ratio=0.2, random_state=RANDOM_STATE, max_iter=20000)),
            ]),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        "RandomForest": Pipeline([
            ("preprocess", preprocess),
            ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        "ExtraTrees": Pipeline([
            ("preprocess", preprocess),
            ("model", ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        "HistGradientBoosting": Pipeline([
            ("preprocess", preprocess),
            ("model", HistGradientBoostingRegressor(max_iter=300, learning_rate=0.04, l2_regularization=0.1, random_state=RANDOM_STATE)),
        ]),
    }


cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}


def evaluate_feature_set(feature_set_name, models, X_train, X_test, y_train, y_test):
    rows = []
    fitted = {}
    for model_name, model in models.items():
        print(f"[{feature_set_name}] Training {model_name}...")
        scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
        model.fit(X_train, y_train)
        fitted[model_name] = model
        pred = model.predict(X_test)
        metrics = evaluate_predictions(y_test, pred)
        rows.append({
            "FeatureSet": feature_set_name,
            "Model": model_name,
            "CV_MAE_mean": -scores["test_MAE"].mean(),
            "CV_MAE_std": scores["test_MAE"].std(),
            "CV_RMSE_mean": -scores["test_RMSE"].mean(),
            "CV_R2_mean": scores["test_R2"].mean(),
            "Holdout_MAE": metrics["MAE"],
            "Holdout_RMSE": metrics["RMSE"],
            "Holdout_R2": metrics["R2"],
            "Holdout_MAPE": metrics["MAPE"],
        })
    result = pd.DataFrame(rows).sort_values("Holdout_MAE")
    return result, fitted


baseline_models = build_models(baseline_preprocess)
baseline_results, baseline_fitted = evaluate_feature_set(
    "BaselineOriginal",
    baseline_models,
    X_base_train,
    X_base_test,
    y_train,
    y_test,
)
display(baseline_results)
baseline_results.to_csv(OUT_DIR / "notebook_baseline_original_results.csv", index=False)

engineered_models = build_models(engineered_preprocess)
engineered_results, engineered_fitted = evaluate_feature_set(
    "EngineeredFeatures",
    engineered_models,
    X_eng_train,
    X_eng_test,
    y_train,
    y_test,
)
display(engineered_results)
engineered_results.to_csv(OUT_DIR / "notebook_engineered_feature_results.csv", index=False)

[BaselineOriginal] Training DummyMedian...


[BaselineOriginal] Training RidgeLogTarget...

[BaselineOriginal] Training ElasticNetLogTarget...

[BaselineOriginal] Training RandomForest...


[BaselineOriginal] Training ExtraTrees...


...

[EngineeredFeatures] Training DummyMedian...
[EngineeredFeatures] Training RidgeLogTarget...


[EngineeredFeatures] Training ElasticNetLogTarget...
[EngineeredFeatures] Training RandomForest...


[EngineeredFeatures] Training ExtraTrees...


[EngineeredFeatures] Training HistGradientBoosting...


,FeatureSet,Model,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_R2_mean,Holdout_MAE,Holdout_RMSE,Holdout_R2,Holdout_MAPE
2,EngineeredFeatures,ElasticNetLogTarget,123321.986810,14887.322963,159141.087626,0.759497,116206.773608,162240.893031,0.821994,0.155569
1,EngineeredFeatures,RidgeLogTarget,140686.949580,17154.485372,188000.507287,0.686119,125005.038857,178376.004058,0.784828,0.159028
3,EngineeredFeatures,RandomForest,177463.757921,24865.482413,231124.781017,0.517106,164901.806452,241525.792964,0.605506,0.216597
5,EngineeredFeatures,HistGradientBoosting,271651.031178,17219.446144,348225.094482,-0.091499,196069.998530,275007.630679,0.488550,0.292564
4,EngineeredFeatures,ExtraTrees,166476.412500,20394.644244,218401.558591,0.578632,206087.916667,349844.416692,0.172318,0.279501
0,EngineeredFeatures,DummyMedian,257731.250000,33086.313494,359455.535575,-0.079574,274500.000000,396660.371099,-0.064023,0.338728


## 8. Compare Feature Sets And Save Artifacts

In [13]:
all_results = pd.concat([baseline_results, engineered_results], ignore_index=True)
best_by_set = all_results.sort_values("Holdout_MAE").groupby("FeatureSet", as_index=False).first()

baseline_best_mae = best_by_set.loc[best_by_set["FeatureSet"] == "BaselineOriginal", "Holdout_MAE"].iloc[0]
best_by_set["Delta_MAE_vs_BaselineBest"] = best_by_set["Holdout_MAE"] - baseline_best_mae
best_by_set["Delta_MAE_pct_vs_BaselineBest"] = best_by_set["Delta_MAE_vs_BaselineBest"] / baseline_best_mae * 100

metric_cols = ["CV_MAE_mean", "CV_MAE_std", "CV_RMSE_mean", "CV_R2_mean", "Holdout_MAE", "Holdout_RMSE", "Holdout_R2", "Holdout_MAPE"]
all_results[metric_cols] = all_results[metric_cols].round(4)
best_by_set[metric_cols + ["Delta_MAE_vs_BaselineBest", "Delta_MAE_pct_vs_BaselineBest"]] = best_by_set[metric_cols + ["Delta_MAE_vs_BaselineBest", "Delta_MAE_pct_vs_BaselineBest"]].round(4)

display(best_by_set)
all_results.to_csv(OUT_DIR / "notebook_feature_set_comparison.csv", index=False)
best_by_set.to_csv(OUT_DIR / "notebook_feature_set_best_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = all_results.sort_values(["Model", "FeatureSet"])
sns.barplot(data=plot_df, x="Model", y="Holdout_MAE", hue="FeatureSet", ax=ax)
ax.set_title("Holdout MAE: Original Features vs Engineered Features")
ax.set_xlabel("")
ax.set_ylabel("MAE (VND)")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(OUT_DIR / "notebook_feature_set_comparison.png", dpi=140, bbox_inches="tight")
plt.show()

best_baseline_name = baseline_results.iloc[0]["Model"]
best_engineered_name = engineered_results.iloc[0]["Model"]
joblib.dump(baseline_fitted[best_baseline_name], MODEL_DIR / "notebook_best_baseline_original_model.pkl")
joblib.dump(engineered_fitted[best_engineered_name], MODEL_DIR / "notebook_best_engineered_feature_model.pkl")
print("Best baseline:", best_baseline_name)
print("Best engineered:", best_engineered_name)


def plot_prediction_diagnostics(model, X_test, y_test, title, output_name):
    pred = model.predict(X_test)
    residuals = y_test - pred

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(y_test / 1e6, pred / 1e6, alpha=0.75)
    mn = min(y_test.min(), pred.min()) / 1e6
    mx = max(y_test.max(), pred.max()) / 1e6
    axes[0].plot([mn, mx], [mn, mx], "r--")
    axes[0].set_title(f"Actual vs Predicted - {title}")
    axes[0].set_xlabel("Actual price (million VND)")
    axes[0].set_ylabel("Predicted price (million VND)")

    axes[1].scatter(pred / 1e6, residuals / 1e6, alpha=0.75)
    axes[1].axhline(0, color="red", linestyle="--")
    axes[1].set_title("Residuals")
    axes[1].set_xlabel("Predicted price (million VND)")
    axes[1].set_ylabel("Residual (million VND)")

    plt.tight_layout()
    plt.savefig(OUT_DIR / output_name, dpi=140, bbox_inches="tight")
    plt.show()


plot_prediction_diagnostics(
    baseline_fitted[best_baseline_name],
    X_base_test,
    y_test,
    f"BaselineOriginal / {best_baseline_name}",
    "notebook_prediction_diagnostics_baseline.png",
)
plot_prediction_diagnostics(
    engineered_fitted[best_engineered_name],
    X_eng_test,
    y_test,
    f"EngineeredFeatures / {best_engineered_name}",
    "notebook_prediction_diagnostics_engineered.png",
)

,FeatureSet,Model,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_R2_mean,Holdout_MAE,Holdout_RMSE,Holdout_R2,Holdout_MAPE,Delta_MAE_vs_BaselineBest,Delta_MAE_pct_vs_BaselineBest
0,BaselineOriginal,ElasticNetLogTarget,139777.3095,10871.7599,188699.6724,0.6877,129106.5981,173990.9128,0.7953,0.1633,0.0000,0.0000
1,EngineeredFeatures,ElasticNetLogTarget,123321.9868,14887.3230,159141.0876,0.7595,116206.7736,162240.8930,0.8220,0.1556,-12899.8245,-9.9916


Best baseline: ElasticNetLogTarget
Best engineered: ElasticNetLogTarget


## 9. Write Evaluation Report

In [14]:
def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    table = df[columns].copy()
    for col in table.columns:
        if pd.api.types.is_numeric_dtype(table[col]):
            if "R2" in col or "MAPE" in col or "pct" in col.lower():
                table[col] = table[col].map(lambda x: f"{x:.4f}")
            else:
                table[col] = table[col].map(lambda x: f"{x:,.0f}")
    header = "| " + " | ".join(table.columns) + " |"
    sep = "| " + " | ".join(["---"] * len(table.columns)) + " |"
    rows = ["| " + " | ".join(str(v) for v in row) + " |" for row in table.to_numpy()]
    return "\n".join([header, sep] + rows)


baseline_best = best_by_set.loc[best_by_set["FeatureSet"] == "BaselineOriginal"].iloc[0]
engineered_best = best_by_set.loc[best_by_set["FeatureSet"] == "EngineeredFeatures"].iloc[0]
delta_mae = engineered_best["Holdout_MAE"] - baseline_best["Holdout_MAE"]
delta_pct = delta_mae / baseline_best["Holdout_MAE"] * 100
comparison_sentence = (
    f"EngineeredFeatures v2 improved holdout MAE by {-delta_mae:,.0f} VND ({-delta_pct:.2f}%)."
    if delta_mae < 0
    else f"EngineeredFeatures v2 worsened holdout MAE by {delta_mae:,.0f} VND ({delta_pct:.2f}%)."
)

report = f"""# Wine Price Model Evaluation

Generated by `notebooks/wine_price_eda_training.ipynb`.

## Setup

- Dataset source: `data/raw_wines.csv`
- Rows after target filtering: `{len(model_df)}`
- Train/Test split: `{len(train_idx)}` train rows / `{len(test_idx)}` test rows
- Split strategy: stratified by 5 quantile bins of `price_vnd`
- CV strategy: 5-fold shuffled KFold on the training split
- Target: `price_vnd`

## Pipelines

- `BaselineOriginal`: all usable original features: numeric fields, nominal categories, and original text from `product_name + short_description`; excludes `url`, `sku`, and `image_url`.
- `EngineeredFeatures`: slim v2 feature set. It keeps stable transforms (`log_volume`, `wine_age`, `log_rating_count`, `bayesian_rating`, `rating_x_popularity`, `alcohol_diff_from_median`), low-cardinality bins (`rating_bin`, `age_group`, `alcohol_bin`, `grape_color_group`), missingness flags, and light text features.
- Removed from the previous engineered attempt: high-cardinality interactions (`country_x_type`, `brand_x_type`, `country_x_grape`, etc.), group target-stat encodings, manual price tiers, and duplicate binary category flags.
- Target-derived alternatives such as `price_per_750ml` and `log_price_per_750ml` are not used as predictors because they contain the target.

## Best Model By Pipeline

{markdown_table(best_by_set, ["FeatureSet", "Model", "CV_MAE_mean", "CV_RMSE_mean", "CV_R2_mean", "Holdout_MAE", "Holdout_RMSE", "Holdout_R2", "Holdout_MAPE", "Delta_MAE_vs_BaselineBest", "Delta_MAE_pct_vs_BaselineBest"])}

## BaselineOriginal Results

{markdown_table(baseline_results, ["Model", "CV_MAE_mean", "CV_RMSE_mean", "CV_R2_mean", "Holdout_MAE", "Holdout_RMSE", "Holdout_R2", "Holdout_MAPE"])}

## EngineeredFeatures Results

{markdown_table(engineered_results, ["Model", "CV_MAE_mean", "CV_RMSE_mean", "CV_R2_mean", "Holdout_MAE", "Holdout_RMSE", "Holdout_R2", "Holdout_MAPE"])}

## Summary

- Best baseline model: `{baseline_best['Model']}` with holdout MAE `{baseline_best['Holdout_MAE']:,.0f}` VND and R2 `{baseline_best['Holdout_R2']:.4f}`.
- Best engineered model: `{engineered_best['Model']}` with holdout MAE `{engineered_best['Holdout_MAE']:,.0f}` VND and R2 `{engineered_best['Holdout_R2']:.4f}`.
- {comparison_sentence}
- Because the dataset has only about 200 rows, treat holdout improvements as directional; prefer changes that also improve CV metrics.

## Generated Artifacts

- `outputs/notebook_baseline_original_results.csv`
- `outputs/notebook_engineered_feature_results.csv`
- `outputs/notebook_feature_set_comparison.csv`
- `outputs/notebook_feature_set_best_summary.csv`
- `outputs/notebook_feature_set_comparison.png`
- `outputs/notebook_prediction_diagnostics_baseline.png`
- `outputs/notebook_prediction_diagnostics_engineered.png`
- `models/notebook_best_baseline_original_model.pkl`
- `models/notebook_best_engineered_feature_model.pkl`
"""

(DOCS_DIR / "evaluation.md").write_text(report, encoding="utf-8")
print("Saved:", DOCS_DIR / "evaluation.md")

Saved: D:\tai ve\VISUAL CODE STUDIO\ML-20252\ML_20252\docs\evaluation.md


## 10. Conclusions And Next Steps

- The target is `price_vnd`, so this notebook is a tabular regression workflow, not a forecasting workflow.
- `BaselineOriginal` shows performance using all usable original features: numeric fields, nominal categories, and original text.
- `EngineeredFeatures` now uses a slim v2 feature set: stable numeric transforms, low-cardinality bins, missingness flags, and light text features.
- Sparse high-cardinality interactions and target-stat encodings were removed because they overfit on the 200-row dataset.
- Compare `Delta_MAE_vs_BaselineBest`: negative means engineered features improved MAE; positive means they hurt holdout performance.
- Target-derived alternatives like `price_per_750ml` are not used as predictors because they contain the target; they can be used only as analysis targets in a separate experiment.